# Dependencies (for venv)

In [97]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Generate the dataset

In [98]:
import random
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   
torch.set_default_device(device)

print(device)

cuda


In [99]:
def generate_objects(N=3, block_size=0.2, initial_tower=True):
    # 1. Generate random order for the goals
    idx_goals = [i for i in range(0, N)]
    random.shuffle(idx_goals)

    # 2. Generate initial order for the blocks by inverting the order of goals
    idx_objs = idx_goals[::-1]

    # 3. Generate goals
    # https://www.generationrobots.com/media/panda-franka-emika-datasheet.pdf
    # according to the datasheet, the Franka Panda can reach in an area of 0.855 [m] around itself
    max_reach_dist = 0.7 # [m]

    # select a random (x,y) position for the goal.
    x_goal = random.random() * max_reach_dist
    y_goal = random.random() * max_reach_dist

    pos_goals = [(x_goal, y_goal, i*block_size) for i in range(0, N)]

    # Generate initial positions
    if initial_tower:
        x_obj = random.random() * max_reach_dist
        y_obj = random.random() * max_reach_dist

        pos_objects = [(x_obj, y_obj, i*block_size) for i in range(0, N)]
    else:
        pos_objects = [(random.random() * max_reach_dist, random.random() * max_reach_dist, i*block_size) for i in range(0, N)]

    goals = {}
    objects = {}
    for i in range(0, N):
        goals[idx_goals[i]] = pos_goals[i]
        objects[idx_objs[i]] = pos_objects[i]

    print(f"goals (in desired order of final stacking):\n{goals}")
    print(f"initial object positions (in order of stacking):\n{objects}")

    return objects, goals, idx_objs, idx_goals


In [100]:
# Now solve the stacking problem by taking the highest block in the objects tower (last in order)
# and putting it into the lowest available goal (first in order)

# build the state vector, ordered nodes with categorical, position and binary features, followed by ordered goals with the same features
def build_state(objects, goals):
    s = []
    N = len(objects)
    for n in range(0, N):
        s.append(0) # objects are all blocks (one hot encode)
        s.append(1) # objects are all blocks (one hot encode)


        s.append(int(objects[n] == goals[n])) # whether block is in goal

        
        for o in objects[n]: s.append(o)
        # s.append(objects[n]) # position of block

    for n in range(0, N):
        s.append(1) # objects are all blocks (one hot encode)
        s.append(0) # objects are all blocks (one hot encode)


        s.append(int(objects[n] == goals[n])) # whether block is in goal


        # s.append(goals[n]) # position of block
        for o in goals[n]: s.append(o)

    return s

def build_example(N=3, block_size=0.2, initial_tower=True):
    objects, goals, idx_objs, idx_goals = generate_objects(N, block_size, initial_tower)
    
    tau = []    
    tau.append(build_state(objects, goals))

    for n in range(0, N):
        # print(f"step #{n}")
        
        o = N-1-n

        # print(f"choosing object #{idx_objs[o]} with pos {objects[idx_objs[o]]}")
        # print(f"moving object #{idx_objs[o]} to goal{goals[idx_objs[o]]}")

        objects[idx_objs[o]] = goals[idx_objs[o]]
        tau.append([o, o])

        # print(f"Updated object positions with goals:")
        # print(f"goals:\n{goals}")
        # print(f"current objects pos:\n{objects}")
        # print("\n")

        tau.append(build_state(objects, goals))

    # TODO (?): add the empty action to a complete graph
    print(tau)
    return tau

In [101]:
# Alternatevely
# at each step, find the highest block that is not yet in its goal
# and move it to the lowest available goal
# repeat until each blocks is in its goal

# Observation to graph

In [102]:
# Now put the demonstration in graph format, using pytorch geometric

import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.utils.convert import to_networkx
import networkx as nx

def build_dataset(N=3, N_EXAMPLES=50):
    data_list = []
    for i in range(N_EXAMPLES):
        print(f"Building example #{i}")
        tau = build_example(N)

        states = tau[0:-1:2] # exclude the complete graph
        actions = tau[1::2]

        print(len(states), states)
        print(len(states), actions)

        # build edges, only once
        # graphs are always fully connected, undirected, and with no edge features and no self loops
        edges = torch.tensor([(i,j) for i in range(N*2) for j in range(N*2) if i != j]).t().contiguous()

        # build nodes and labels
        for n in range(0, N):
            s = states[n]
            a = actions[n]

            # torch_geometric.data.Data is a single graph
            # x is the input graph, as a matrix of dimension [node, node_features]
            # I have N*2 nodes (N blocks, N goals) and 5 features each
            x = torch.reshape(torch.tensor(s), (N*2, -1)) # N*2 considers both goals and blocks
            # y is divided into two vectors, one hot encoded, probabilities for the block and the goal
            # y = torch.Tensor.float(F.one_hot(torch.tensor(a), num_classes=N).t())
            # y = (y[:, 0].reshape(-1), y[:, 1].reshape(-1))

            y = torch.tensor(a)
            G = torch_geometric.data.Data(x=x, edge_index=edges, y=y)

            # save all the graphs into a single data_list for use as a DataLoader later
            data_list.append(G)

        print(data_list)
    return data_list

# Build torch_geometric dataset from examples

In [103]:
# https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html#frequently-asked-questions
# just create a DataLoader starting from the list of example Data
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

train_dataset = build_dataset(N=3, N_EXAMPLES=15) + build_dataset(N=4, N_EXAMPLES=15)

print("Training dataset: ", train_dataset)

# it actually trains better (as in lower loss) with a batch_size of 1
train_loader = DataLoader(train_dataset, batch_size=1)
val_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10), batch_size=1)
# test_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=20), batch_size=1)

Building example #0
goals (in desired order of final stacking):
{2: (0.6707719216459553, 0.06643559493368693, 0.0), 0: (0.6707719216459553, 0.06643559493368693, 0.2), 1: (0.6707719216459553, 0.06643559493368693, 0.4)}
initial object positions (in order of stacking):
{1: (0.33357381754502474, 0.18745297723706233, 0.0), 0: (0.33357381754502474, 0.18745297723706233, 0.2), 2: (0.33357381754502474, 0.18745297723706233, 0.4)}
[[0, 1, 0, 0.33357381754502474, 0.18745297723706233, 0.2, 0, 1, 0, 0.33357381754502474, 0.18745297723706233, 0.0, 0, 1, 0, 0.33357381754502474, 0.18745297723706233, 0.4, 1, 0, 0, 0.6707719216459553, 0.06643559493368693, 0.2, 1, 0, 0, 0.6707719216459553, 0.06643559493368693, 0.4, 1, 0, 0, 0.6707719216459553, 0.06643559493368693, 0.0], [2, 2], [0, 1, 0, 0.33357381754502474, 0.18745297723706233, 0.2, 0, 1, 0, 0.33357381754502474, 0.18745297723706233, 0.0, 0, 1, 1, 0.6707719216459553, 0.06643559493368693, 0.0, 1, 0, 0, 0.6707719216459553, 0.06643559493368693, 0.2, 1, 0, 0, 

# Build NN

In [104]:
# https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html
# https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html
# https://colab.research.google.com/github/Alireza-Akhavan/graph-neural-network/blob/main/07-Graph_Classification_pyg.ipynb#scrollTo=CN3sRVuaQ88l
# https://khairulislam.github.io/GNN/pages/graph/Graph_Classification.html

from torch.nn import Linear
from torch_geometric.nn import DenseSAGEConv
from torch_geometric.utils import to_dense_adj, to_dense_batch

import lightning as L
from lightning.pytorch.callbacks import Callback, EarlyStopping
from lightning.pytorch.callbacks.early_stopping import EarlyStoppingReason

class GCN(torch.nn.Module):
    def __init__(self, hidden_nodes=64):
        super().__init__()
        # All policies consist of 3 hidden layers, with 64 hidden units each and ReLU activation.
        # For attention policies, the number of attention heads were set to 1.
        self.conv1 = DenseSAGEConv(6, hidden_nodes)
        self.conv2 = DenseSAGEConv(hidden_nodes, hidden_nodes)
        self.conv3 = DenseSAGEConv(hidden_nodes, hidden_nodes)
        self.lin1 = Linear(hidden_nodes, hidden_nodes)
        self.lin2 = Linear(hidden_nodes, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        # print(x, edge_index, batch)

        # batch = to_dense_batch(batch)
        adj = to_dense_adj(edge_index)

        # print(adj, adj.shape)

        x = self.conv1(x, adj)
        x = F.relu(x)
        x = self.conv2(x, adj)
        x = F.relu(x)
        x = self.conv3(x, adj)
        x = F.relu(x)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.lin2(x)

        # print()
        # print("pre softmax ", x, x.shape)

        # The output of the GNN policy is K + L dimensional, reshaped into K and L dimensional outputs
        objects = x[:, :data.num_nodes//2]
        goals = x[:, data.num_nodes//2:]

        return objects, goals
        # return F.softmax(objects, dim=1), F.softmax(goals, dim=1)

model = GCN(hidden_nodes=16).to(device)
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=5e-6)

model.train()
for epoch in range(5000):
    running_loss = 0.
    for i,data in enumerate(train_loader):
        optimizer.zero_grad()
        objects, goals = model(data)
        objects = objects.squeeze()
        goals = goals.squeeze()
        # print(objects, objects.shape, goals, goals.shape)
        # print(data.y, data.y.shape)
        # print(data.y[0].shape)
        l1 = F.cross_entropy(objects, data.y[0])
        l2 = F.cross_entropy(goals, data.y[1])
        loss = l1+l2
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f'Loss (CE): {running_loss}')

    correct_obj = 0.
    correct_goal = 0.
    for i,data in enumerate(train_dataset):
        objects, goals = model(data)
        objects = objects.squeeze()
        goals = goals.squeeze()

        # print(torch.argmax(objects)==data.y[0], torch.argmax(objects), data.y[0])

        correct_obj += (torch.argmax(objects) == data.y[0]).float()
        correct_goal += (torch.argmax(goals) == data.y[1]).float()


    acc_obj = correct_obj / i
    acc_goal = correct_goal / i
    print(f'Accuracy (objects/goals): {acc_obj:.4f}/{acc_goal:.4f}')

Loss (CE): 265.1770339012146
Accuracy (objects/goals): 0.2019/0.2981
Loss (CE): 265.16624188423157
Accuracy (objects/goals): 0.2404/0.2981
Loss (CE): 265.159211397171
Accuracy (objects/goals): 0.2692/0.2981
Loss (CE): 265.1468737125397
Accuracy (objects/goals): 0.2692/0.2981
Loss (CE): 265.14388132095337
Accuracy (objects/goals): 0.2788/0.2981
Loss (CE): 265.13657307624817
Accuracy (objects/goals): 0.2692/0.2981
Loss (CE): 265.1313998699188
Accuracy (objects/goals): 0.2692/0.2981
Loss (CE): 265.1233377456665
Accuracy (objects/goals): 0.2788/0.2981
Loss (CE): 265.11805391311646
Accuracy (objects/goals): 0.2981/0.2981
Loss (CE): 265.1068480014801
Accuracy (objects/goals): 0.2981/0.2981
Loss (CE): 265.09880805015564
Accuracy (objects/goals): 0.2981/0.2981
Loss (CE): 265.08944964408875
Accuracy (objects/goals): 0.2981/0.3077
Loss (CE): 265.0891795158386
Accuracy (objects/goals): 0.3077/0.3077
Loss (CE): 265.07932472229004
Accuracy (objects/goals): 0.3365/0.3077
Loss (CE): 265.0715496540069

KeyboardInterrupt: 